In [1]:
import polars as pl
df = pl.read_csv("/home/rohan/agn_photometry/photometry/results/Target_Table.csv")

In [2]:
df.head()

Name of the source,RA,DEC,redshift,FIBER,MJD,PLATE,Link
str,f64,f64,f64,i64,i64,i64,str
"""152942.19+350851.3""",232.425832,35.147586,0.287,490,58198,10742,"""https://skyserver.sdss.org/dr1…"
"""075005.28+292944.3""",117.522012,29.495662,0.327794,280,52636,1060,"""https://skyserver.sdss.org/dr1…"
"""095919.14+090659.4""",149.829764,9.116513,0.560051,210,52999,1307,"""https://skyserver.sdss.org/dr1…"
"""150348.33+193941.3""",225.951406,19.661489,0.3374,463,54556,2792,"""https://skyserver.sdss.org/dr1…"
"""092158.92+034235.7""",140.495517,3.709927,0.324812,629,52252,567,"""https://skyserver.sdss.org/dr1…"


In [ ]:
from alerce.core import Alerce

alerce = Alerce()


result = alerce.query_objects(ra=232.425832, dec=35.147586, radius=5, format="pandas")

print(result)
print(result.columns)
print(result.shape)

            oid ndethist  ncovhist  mjdstarthist    mjdendhist  corrected  \
0  ZTF18aasdikm     1238      2006  58247.319363  61077.508333       True   

   stellar  ndet g_r_max g_r_max_corr  ...       lastmjd      deltajd  \
0    False   478    None         None  ...  61077.508333  2823.105312   

       meanra    meandec   sigmara  sigmadec  class  classifier  probability  \
0  232.425896  35.147581  0.004028  0.003293   None        None         None   

     step_id_corr  
0  27.5.7a32.dev1  

[1 rows x 23 columns]
Index(['oid', 'ndethist', 'ncovhist', 'mjdstarthist', 'mjdendhist',
       'corrected', 'stellar', 'ndet', 'g_r_max', 'g_r_max_corr', 'g_r_mean',
       'g_r_mean_corr', 'firstmjd', 'lastmjd', 'deltajd', 'meanra', 'meandec',
       'sigmara', 'sigmadec', 'class', 'classifier', 'probability',
       'step_id_corr'],
      dtype='object')
(1, 23)


In [ ]:
oid = result["oid"][0]  

detections = alerce.query_detections(oid, format="pandas")

print(detections)
print(detections.columns)
print(detections.shape)

     tid           mjd               candid  fid            pid  diffmaglim  \
0    ztf  58254.403021   500403024415015002    1   500403024415   20.884900   
1    ztf  58278.226979   524226974415015001    1   524226974415   20.860466   
2    ztf  58278.267731   524267734415015000    2   524267734415   20.661264   
3    ztf  58281.212685   527212684415015001    2   527212684415   20.785824   
4    ztf  58288.274005   534274004415015001    2   534274004415   20.632570   
..   ...           ...                  ...  ...            ...         ...   
473  ztf  61060.518657  3306518654415015000    2  3306518654415   20.581257   
474  ztf  61060.530289  3306530284415015000    1  3306530284415   20.615260   
475  ztf  61065.471296  3311471294415015000    2  3311471294415   20.273010   
476  ztf  61071.486505  3317486504415015001    1  3317486504415   19.421251   
477  ztf  61077.508333  3323508334415015000    1  3323508334415   20.077948   

     isdiffpos   nid    distnr     magpsf  ...  sig

In [ ]:
import requests
import re
import time
import io

with open("/home/rohan/atlas_creds.txt") as f:
    lines = f.read().strip().split("\n")
    username = lines[0]
    password = lines[1]

BASEURL = "https://fallingstar-data.com/forcedphot"

resp = requests.post(
    url=f"{BASEURL}/api-token-auth/",
    data={"username": username, "password": password}
)

if resp.status_code == 200:
    token = resp.json()["token"]
    headers = {"Authorization": f"Token {token}", "Accept": "application/json"}
    print("Auth successful")
else:
    print(f"Auth failed: {resp.status_code}")
    print(resp.json())

Auth successful


In [ ]:
ra = 232.425832
dec = 35.147586

task_url = None
while not task_url:
    resp = requests.post(
        f"{BASEURL}/queue/",
        headers=headers,
        data={"ra": ra, "dec": dec}
    )
    if resp.status_code == 201:
        task_url = resp.json()["url"]
        print(f"Job submitted: {task_url}")
    elif resp.status_code == 429:
        message = resp.json()["detail"]
        t_sec = re.findall(r"(\d+) seconds", message)
        waittime = int(t_sec[0]) if t_sec else 10
        print(f"Rate limited, waiting {waittime}s")
        time.sleep(waittime)
    else:
        print(f"Error: {resp.status_code}")
        print(resp.json())
        break

Job submitted: https://fallingstar-data.com/forcedphot/queue/4380768/


In [ ]:
result_url = None
while not result_url:
    resp = requests.get(task_url, headers=headers)
    if resp.status_code == 200:
        data = resp.json()
        if data.get("finishtimestamp"):
            result_url = data["result_url"]
            print(f"Job complete: {result_url}")
            break
        else:
            print("Still running...")
            time.sleep(10)
    else:
        print(f"Error: {resp.status_code}")
        break

Still running...
Still running...
Still running...
Job complete: https://fallingstar-data.com/forcedphot/static/results/job4380768.txt


In [ ]:
import polars as pl
import time

alerce_dfs = []

print(f"Starting ALERCE processing for {df.height} sources to Parquet...\n")

for row in df.iter_rows(named=True):
    source_name = row["Name of the source"]
    target_ra = row["RA"]
    target_dec = row["DEC"]
    
    print(f"Querying ALERCE for {source_name}... ", end="")
    try:
        obj_match = alerce.query_objects(ra=target_ra, dec=target_dec, radius=5, format="pandas")
        
        if obj_match.empty:
            print("No matching ZTF source found within 5 arcsec.")
            continue
            
        ztf_oid = obj_match["oid"].iloc[0]
        
        detections = alerce.query_detections(ztf_oid, format="pandas")
        
        if detections.empty:
            print(f"Object found ({ztf_oid}) but contains no detections.")
            continue
            
        keep_cols = ['tid', 'mjd', 'magpsf_corr', 'sigmapsf', 'fid']
        minimal_df = detections[keep_cols].copy()
        
        minimal_df['source_name'] = source_name
        minimal_df['RA'] = target_ra
        minimal_df['DEC'] = target_dec
        
        alerce_dfs.append(pl.from_pandas(minimal_df))
        print(f"Success! Collected {len(minimal_df)} data points.")
        
    except Exception as e:
        print(f"Skipped due to API error: {e}")
    
    time.sleep(0.5)

if alerce_dfs:
    final_alerce_df = pl.concat(alerce_dfs)
    output_parquet = "/home/rohan/agn_photometry/photometry/results/alerce_photometry.parquet"
    final_alerce_df.write_parquet(output_parquet, compression="snappy")
    print(f"\n[ALERCE Done] Saved master Parquet with {final_alerce_df.height} rows to {output_parquet}")
else:
    print("\n[ALERCE Done] No data collected.")

Starting ALERCE processing for 32 sources to Parquet...

Querying ALERCE for 152942.19+350851.3... Success! Collected 478 data points.
Querying ALERCE for 075005.28+292944.3... Success! Collected 25 data points.
Querying ALERCE for 095919.14+090659.4... No matching ZTF source found within 5 arcsec.
Querying ALERCE for 150348.33+193941.3... Success! Collected 34 data points.
Querying ALERCE for 092158.92+034235.7... Success! Collected 52 data points.
Querying ALERCE for 164331.90+304835.5... No matching ZTF source found within 5 arcsec.
Querying ALERCE for 083325.69+391204.7... Success! Collected 5 data points.
Querying ALERCE for 001255.59+014750.9... Success! Collected 352 data points.
Querying ALERCE for 154535.87+014958.1... No matching ZTF source found within 5 arcsec.
Querying ALERCE for 093856.54+435415.0... Success! Collected 4 data points.
Querying ALERCE for 172052.29+590153.7... No matching ZTF source found within 5 arcsec.
Querying ALERCE for 205601.38-062049.7... No matchin

In [ ]:
import io
import re
import time
import requests
import pandas as pd
import polars as pl

atlas_dfs = []

print(f"Starting Production ATLAS Parquet Pipeline for {df.height} sources...")

for row in df.iter_rows(named=True):
    source_name = row["Name of the source"]
    target_ra = row["RA"]
    target_dec = row["DEC"]
    
    print(f"\n--- Processing ATLAS Target: {source_name} ---")
    
    task_url = None
    while not task_url:
        resp = requests.post(
            f"{BASEURL}/queue/",
            headers=headers,
            data={"ra": target_ra, "dec": target_dec}
        )
        if resp.status_code == 201:
            task_url = resp.json()["url"]
            print(f" Job successfully queued: {task_url}")
        elif resp.status_code == 429:
            message = resp.json().get("detail", "")
            t_sec = re.findall(r"available in (\d+) seconds", message)
            t_min = re.findall(r"available in (\d+) minutes", message)
            waittime = int(t_sec[0]) if t_sec else int(t_min[0]) * 60 if t_min else 15
            print(f" Rate limited. Sleeping for {waittime}s...")
            time.sleep(waittime)
        else:
            print(f" Failed submission: {resp.status_code} - {resp.json()}")
            break
            
    if not task_url:
        continue

    result_url = None
    fail_tracker = 0
    while not result_url:
        resp = requests.get(task_url, headers=headers)
        if resp.status_code == 200:
            job_data = resp.json()
            if job_data.get("finishtimestamp"):
                result_url = job_data["result_url"]
                print(" Processing complete. Downloading file...")
                break
            else:
                time.sleep(10)
        else:
            fail_tracker += 1
            if fail_tracker > 5:
                break
            time.sleep(10)
            
    if not result_url:
        continue

    # Step 3: Parse and Handle Potential Formatting Anomalies
    try:
        raw_text = requests.get(result_url, headers=headers).text
        
        # Split lines to clean out the messy headers/comments safely
        lines = raw_text.split('\n')
        data_lines = []
        headers_line = None
        
        for line in lines:
            line = line.strip()
            if not line:
                continue
            if line.startswith('###') or line.startswith('#'):
                # Strip symbols to check if it's the header row
                cleaned_header = line.replace('###', '').replace('#', '').strip()
                if cleaned_header.startswith('MJD'):
                    headers_line = re.sub(r'\s+', ',', cleaned_header)
                continue
            # If it starts with a digit (the MJD float), it's a valid numerical data row
            if re.match(r'^\d', line):
                data_lines.append(re.sub(r'\s+', ',', line))

        if not headers_line or not data_lines:
            print(f" Error: No readable structural data lines found for {source_name}")
            continue

        # Reconstruct into clean, structured CSV-like stream
        csv_data = headers_line + '\n' + '\n'.join(data_lines)
        temp_pd = pd.read_csv(io.StringIO(csv_data), sep=',')
        
        # Force column name normalization
        temp_pd.columns = [c.strip() for c in temp_pd.columns]
        
        # Standardize core columns
        keep_cols = ['MJD', 'm', 'dm', 'F']
        minimal_df = temp_pd[keep_cols].copy()
        
        # Append meta coordinates and source identities
        minimal_df['source_name'] = source_name
        minimal_df['RA'] = target_ra
        minimal_df['DEC'] = target_dec
        
        atlas_dfs.append(pl.from_pandas(minimal_df))
        print(f" Success! Extracted {len(minimal_df)} forced photometry entries.")
        
    except Exception as e:
        print(f" Critical breakdown parsing data for {source_name}: {e}")

if atlas_dfs:
    final_atlas_df = pl.concat(atlas_dfs)
    output_parquet = "/home/rohan/agn_photometry/photometry/results/atlas_photometry.parquet"
    final_atlas_df.write_parquet(output_parquet, compression="snappy")
    print(f"\n[ATLAS Done] Saved master Parquet with {final_atlas_df.height} rows to {output_parquet}")
else:
    print("\n[ATLAS Done] No forced photometry data was retrieved.")

Starting Production ATLAS Parquet Pipeline for 32 sources...

--- Processing ATLAS Target: 152942.19+350851.3 ---
 Job successfully queued: https://fallingstar-data.com/forcedphot/queue/4380771/
 Processing complete. Downloading file...
 Success! Extracted 82 forced photometry entries.

--- Processing ATLAS Target: 075005.28+292944.3 ---
 Job successfully queued: https://fallingstar-data.com/forcedphot/queue/4380773/
 Processing complete. Downloading file...
 Success! Extracted 19 forced photometry entries.

--- Processing ATLAS Target: 095919.14+090659.4 ---
 Job successfully queued: https://fallingstar-data.com/forcedphot/queue/4380774/
 Processing complete. Downloading file...
 Success! Extracted 33 forced photometry entries.

--- Processing ATLAS Target: 150348.33+193941.3 ---
 Job successfully queued: https://fallingstar-data.com/forcedphot/queue/4380775/
 Processing complete. Downloading file...
 Success! Extracted 92 forced photometry entries.

--- Processing ATLAS Target: 09215

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

parquet_alerce = "/home/rohan/agn_photometry/photometry/results/alerce_photometry.parquet"
parquet_atlas = "/home/rohan/agn_photometry/photometry/results/atlas_photometry.parquet"

df_alerce = pl.read_parquet(parquet_alerce)
df_atlas = pl.read_parquet(parquet_atlas)

unique_sources = df["Name of the source"].sort().to_list()

output_pdf_path = "/home/rohan/agn_photometry/photometry/results/agn_unfiltered_lightcurves.pdf"
print(f"Generating raw 4-panel profiles for all {len(unique_sources)} original targets...\n")

with PdfPages(output_pdf_path) as pdf:
    for i, target_id in enumerate(unique_sources, start=1):
        print(f"[{i}/{len(unique_sources)}] Plotting raw panels for: {target_id}")
        
        obj_alerce = df_alerce.filter(pl.col("source_name") == target_id)
        obj_atlas = df_atlas.filter(pl.col("source_name") == target_id)
        
        # Pull coordinates from original target table row
        target_row = df.filter(pl.col("Name of the source") == target_id)
        ra_val = target_row["RA"][0] if not target_row.is_empty() else 0.0
        dec_val = target_row["DEC"][0] if not target_row.is_empty() else 0.0

        mjd_stamps = []
        if not obj_alerce.is_empty():
            mjd_stamps.extend([obj_alerce["mjd"].min(), obj_alerce["mjd"].max()])
        if not obj_atlas.is_empty():
            mjd_stamps.extend([obj_atlas["MJD"].min(), obj_atlas["MJD"].max()])
            
        t_start = min(mjd_stamps) if mjd_stamps else 0.0
        t_end = max(mjd_stamps) if mjd_stamps else 0.0
        total_duration_days = t_end - t_start

        fig, axes = plt.subplots(4, 1, figsize=(11, 11), sharex=True, dpi=95)
        
        filter_config = [
            ("alerce", 1, "ZTF $g$-band", 0, "#2ca02c", "o"),
            ("alerce", 2, "ZTF $r$-band", 1, "#d62728", "o"),
            ("atlas", "c", "ATLAS $c$-band", 2, "#17becf", "s"),
            ("atlas", "o", "ATLAS $o$-band", 3, "#ff7f0e", "s")
        ]
        
        for survey, fid, band_desc, ax_idx, color, marker in filter_config:
            ax = axes[ax_idx]
            
            if survey == "alerce":
                band_data = obj_alerce.filter(pl.col("fid") == fid)
                x_vals, y_vals, y_errs = band_data["mjd"], band_data["magpsf_corr"], band_data["sigmapsf"]
            else:
                band_data = obj_atlas.filter(pl.col("F") == fid)
                x_vals, y_vals, y_errs = band_data["MJD"], band_data["m"], band_data["dm"]
                
            if not band_data.is_empty():
                ax.errorbar(
                    x=x_vals, y=y_vals, yerr=y_errs,
                    fmt=marker, markersize=3.5, color=color, alpha=0.7, 
                    label=f"{band_desc} ({len(band_data)} pts)", ls="none", elinewidth=0.8
                )
                ax.legend(loc="upper right", frameon=True, fontsize=8.5, facecolor="#f8f9fa", edgecolor="#e9ecef")
            else:
                ax.text(0.5, 0.5, f"No coverage available for {band_desc}", transform=ax.transAxes, 
                        ha='center', va='center', color='gray', fontsize=10, style='italic')
            
            ax.invert_yaxis()  
            ax.set_ylabel("PSF Mag", fontsize=9)
            ax.grid(True, linestyle="--", alpha=0.35)
            ax.tick_params(axis='both', labelsize=9.5)

        title_string = f"RAW UNFILTERED PROFILE: {target_id}"
        subtitle_string = (
            f"Coordinates: RA = {ra_val:.6f}° , DEC = {dec_val:.6f}° (J2000)   │   "
            f"Baseline: {total_duration_days:.1f} days"
        )
        
        fig.suptitle(title_string, fontsize=13, fontweight="bold", y=0.98)
        axes[0].set_title(subtitle_string, fontsize=10, color="#495057", pad=12)
        axes[3].set_xlabel("Time Axis (Modified Julian Date [MJD])", fontsize=11, labelpad=8)
        
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        pdf.savefig(fig)
        plt.close(fig)

print(f"\nDone! Raw unfiltered 4-panel catalog saved to:\n{output_pdf_path}")

Generating raw 4-panel profiles for all 32 original targets...

[1/32] Plotting raw panels for: 001255.59+014750.9
[2/32] Plotting raw panels for: 012715.48-001115.1
[3/32] Plotting raw panels for: 022325.01-003612.6
[4/32] Plotting raw panels for: 075005.28+292944.3
[5/32] Plotting raw panels for: 075947.73+112507.3
[6/32] Plotting raw panels for: 082311.26+435318.5
[7/32] Plotting raw panels for: 083325.69+391204.7
[8/32] Plotting raw panels for: 084245.18+445533.9
[9/32] Plotting raw panels for: 090550.30+003948.1
[10/32] Plotting raw panels for: 092158.92+034235.7
[11/32] Plotting raw panels for: 093856.54+435415.0
[12/32] Plotting raw panels for: 094203.33+593709.5
[13/32] Plotting raw panels for: 095746.83+303024.2
[14/32] Plotting raw panels for: 095919.14+090659.4
[15/32] Plotting raw panels for: 110104.14+062829.5
[16/32] Plotting raw panels for: 113618.90+125554.8
[17/32] Plotting raw panels for: 121028.83+025454.2
[18/32] Plotting raw panels for: 132404.20+433407.1
[19/32] P

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

parquet_alerce = "/home/rohan/agn_photometry/photometry/results/alerce_photometry.parquet"
parquet_atlas = "/home/rohan/agn_photometry/photometry/results/atlas_photometry.parquet"

df_alerce = pl.read_parquet(parquet_alerce)
df_atlas = pl.read_parquet(parquet_atlas)

df_alerce = df_alerce.filter(pl.col("magpsf_corr").is_not_null() & (pl.col("magpsf_corr") > 0))
df_atlas = df_atlas.filter(pl.col("m").is_not_null() & (pl.col("m") > 0))

unique_sources = df["Name of the source"].sort().to_list()

output_pdf_path = "/home/rohan/agn_photometry/photometry/results/agn_sigma_filtered_lightcurves.pdf"
print(f"Generating robust data-rich profiles for all {len(unique_sources)} original targets...\n")

with PdfPages(output_pdf_path) as pdf:
    for i, target_id in enumerate(unique_sources, start=1):
        print(f"[{i}/{len(unique_sources)}] Processing focused layout for: {target_id}")
        
        obj_alerce = df_alerce.filter(pl.col("source_name") == target_id)
        obj_atlas = df_atlas.filter(pl.col("source_name") == target_id)
        
        target_row = df.filter(pl.col("Name of the source") == target_id)
        ra_val = target_row["RA"][0] if not target_row.is_empty() else 0.0
        dec_val = target_row["DEC"][0] if not target_row.is_empty() else 0.0

        mjd_stamps = []
        if not obj_alerce.is_empty():
            mjd_stamps.extend([obj_alerce["mjd"].min(), obj_alerce["mjd"].max()])
        if not obj_atlas.is_empty():
            mjd_stamps.extend([obj_atlas["MJD"].min(), obj_atlas["MJD"].max()])
            
        t_start = min(mjd_stamps) if mjd_stamps else 0.0
        t_end = max(mjd_stamps) if mjd_stamps else 0.0
        total_duration_days = t_end - t_start

        fig, axes = plt.subplots(4, 1, figsize=(11, 11), sharex=True, dpi=95)
        
        filter_config = [
            ("alerce", 1, "ZTF $g$-band (Green $\sim$480 nm)", 0, "#2ca02c", "o"),
            ("alerce", 2, "ZTF $r$-band (Red $\sim$640 nm)", 1, "#d62728", "o"),
            ("atlas", "c", "ATLAS $c$-band (Cyan $\sim$420–550 nm)", 2, "#17becf", "s"),
            ("atlas", "o", "ATLAS $o$-band (Orange $\sim$560–820 nm)", 3, "#ff7f0e", "s")
        ]
        
        for survey, fid, band_desc, ax_idx, color, marker in filter_config:
            ax = axes[ax_idx]
            
            if survey == "alerce":
                band_data = obj_alerce.filter(pl.col("fid") == fid)
                x_vals, y_vals, y_errs = band_data["mjd"], band_data["magpsf_corr"], band_data["sigmapsf"]
            else:
                band_data = obj_atlas.filter(pl.col("F") == fid)
                x_vals, y_vals, y_errs = band_data["MJD"], band_data["m"], band_data["dm"]
                
            if not band_data.is_empty():
                ax.errorbar(
                    x=x_vals, y=y_vals, yerr=y_errs,
                    fmt=marker, markersize=3.5, color=color, alpha=0.7, 
                    label=f"{band_desc} ({len(band_data)} pts)", ls="none", elinewidth=0.8
                )
                
                q_low = y_vals.quantile(0.05)
                q_high = y_vals.quantile(0.95)
                padding = max(0.4, (q_high - q_low) * 0.15)
                
                ax.set_ylim(bottom=q_high + padding, top=q_low - padding)
                
                ax.legend(loc="upper right", frameon=True, fontsize=8.5, facecolor="#f8f9fa", edgecolor="#e9ecef")
            else:
                ax.text(0.5, 0.5, f"No coverage available for {band_desc}", transform=ax.transAxes, 
                        ha='center', va='center', color='gray', fontsize=10, style='italic')
            
            ax.set_ylabel("Apparent Brightness\n(PSF Mag [mag])", fontsize=9)
            ax.grid(True, linestyle="--", alpha=0.35)
            ax.tick_params(axis='both', labelsize=9.5)

        title_string = f"TARGET OBJECT PROFILE: {target_id}"
        subtitle_string = (
            f"Coordinates: RA = {ra_val:.6f}° , DEC = {dec_val:.6f}° (J2000)   │   "
            f"Baseline: MJD {t_start:.2f} to {t_end:.2f} ($\Delta t = {total_duration_days:.1f}$ days)"
        )
        
        fig.suptitle(title_string, fontsize=13, fontweight="bold", y=0.98)
        axes[0].set_title(subtitle_string, fontsize=10, color="#495057", pad=12)
        axes[3].set_xlabel("Time Axis (Modified Julian Date [MJD])", fontsize=11, labelpad=8)
        
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        pdf.savefig(fig)
        plt.close(fig)

print(f"\nSuccess! Robust PDF generated across all 32 targets:\n{output_pdf_path}")

Generating robust data-rich profiles for all 32 original targets...

[1/32] Processing focused layout for: 001255.59+014750.9


<>:47: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:48: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:49: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:50: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:91: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<>:47: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:48: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences wil

[2/32] Processing focused layout for: 012715.48-001115.1
[3/32] Processing focused layout for: 022325.01-003612.6
[4/32] Processing focused layout for: 075005.28+292944.3
[5/32] Processing focused layout for: 075947.73+112507.3
[6/32] Processing focused layout for: 082311.26+435318.5
[7/32] Processing focused layout for: 083325.69+391204.7
[8/32] Processing focused layout for: 084245.18+445533.9
[9/32] Processing focused layout for: 090550.30+003948.1
[10/32] Processing focused layout for: 092158.92+034235.7
[11/32] Processing focused layout for: 093856.54+435415.0
[12/32] Processing focused layout for: 094203.33+593709.5
[13/32] Processing focused layout for: 095746.83+303024.2
[14/32] Processing focused layout for: 095919.14+090659.4
[15/32] Processing focused layout for: 110104.14+062829.5
[16/32] Processing focused layout for: 113618.90+125554.8
[17/32] Processing focused layout for: 121028.83+025454.2
[18/32] Processing focused layout for: 132404.20+433407.1
[19/32] Processing foc